# Demo: Medicion Automatizada de Botellas en Imagenes RAW (CR3)

**Objetivo:** Detectar y medir botellas de vidrio en imagenes Canon RAW (CR3),  
comparando un pipeline **baseline** (Canny estandar) contra uno **mejorado**  
(CLAHE + filtro bilateral + morfologia robusta + validacion con keypoints).

**Estructura del notebook:**
1. Imports y configuracion
2. Carga de parametros RAW
3. Carga de imagenes desde `fotos_raw/`
4. Pipeline baseline
5. Pipeline mejorado
6. Extraccion de medidas
7. Evaluacion comparativa
8. Conclusiones

---
## 1. Imports y configuracion

In [ ]:
import sys, os

# Asegurar que el directorio raiz del proyecto este en el path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

from src.raw_loader        import load_config, load_images_from_folder
from src.preprocessing     import baseline_preprocess, improved_preprocess
from src.edge_detection    import (baseline_edges, baseline_mask,
                                    improved_edges, improved_mask,
                                    get_bottle_contour)
from src.contour_measurement import measure_bottle
from src.evaluation        import (run_baseline, run_improved,
                                    build_summary_table, print_stats,
                                    plot_comparison_bar, plot_edge_quality,
                                    plot_contour_overlay)
from src.utils             import (show_pipeline_steps, annotate_measurements,
                                    format_measures_table, save_outputs)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 11

print('Imports OK')
print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version:  {np.__version__}')

---
## 2. Carga de parametros RAW

In [ ]:
CONFIG_PATH  = os.path.join(PROJECT_ROOT, 'config', 'raw_config.json')
IMAGE_FOLDER = os.path.join(PROJECT_ROOT, 'fotos_raw')
OUTPUT_FOLDER = os.path.join(PROJECT_ROOT, 'outputs')
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

config = load_config(CONFIG_PATH)
print('Configuracion cargada:')
for k, v in config.items():
    print(f'  {k}: {v}')

---
## 3. Carga de imagenes desde `fotos_raw/`

In [ ]:
print(f'Buscando imagenes en: {IMAGE_FOLDER}')
images = load_images_from_folder(IMAGE_FOLDER, config)

if not images:
    print('\n[ADVERTENCIA] No se encontraron imagenes RAW.')
    print('  Coloca archivos CR3/CR2/raw en la carpeta fotos_raw/')
else:
    print(f'\nTotal imagenes cargadas: {len(images)}')
    for img_dict in images:
        h, w = img_dict['image'].shape[:2]
        print(f"  {img_dict['name']:30s}  {w}x{h} px")

In [ ]:
# Vista previa de las imagenes cargadas
if images:
    n = min(len(images), 5)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, img_dict in zip(axes, images[:n]):
        ax.imshow(img_dict['image'], cmap='gray')
        ax.set_title(img_dict['name'], fontsize=9)
        ax.axis('off')
    fig.suptitle('Imagenes RAW cargadas (escala de grises)', fontsize=12)
    fig.tight_layout()
    plt.show()

---
## 4. Pipeline Baseline

Pasos:
1. Suavizado Gaussiano
2. Canny con umbrales adaptativos (sigma de la mediana)
3. Cierre morfologico
4. Extraccion del contorno principal

In [ ]:
if not images:
    print('Sin imagenes, omitiendo pipeline baseline.')
else:
    # Demostrar pipeline baseline en la primera imagen
    demo = images[0]
    gray = demo['image']

    prep_b  = baseline_preprocess(gray)
    edges_b = baseline_edges(prep_b)
    mask_b  = baseline_mask(edges_b)

    fig = show_pipeline_steps(gray, prep_b, edges_b, mask_b,
                               title=f"Pipeline BASELINE  —  {demo['name']}")
    plt.show()
    print('Pipeline baseline ejecutado.')

---
## 5. Pipeline Mejorado

Mejoras sobre baseline:
- **CLAHE** para contraste local adaptativo
- **Filtro bilateral** (suaviza ruido, preserva bordes)
- **Supresion de reflejos** (hot-spots)
- **Morfologia robusta** (cierre mas agresivo + dilatacion)
- **Validacion con keypoints** SIFT / ORB fallback

In [ ]:
if not images:
    print('Sin imagenes, omitiendo pipeline mejorado.')
else:
    prep_i  = improved_preprocess(gray)
    edges_i = improved_edges(prep_i)
    mask_i  = improved_mask(edges_i, prep_i)

    fig = show_pipeline_steps(gray, prep_i, edges_i, mask_i,
                               title=f"Pipeline MEJORADO  —  {demo['name']}")
    plt.show()
    print('Pipeline mejorado ejecutado.')

In [ ]:
# Comparacion de preprocesado: Gaussiano vs CLAHE + Bilateral
if images:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(gray, cmap='gray');    axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(prep_b, cmap='gray'); axes[1].set_title('Baseline: Gaussiano'); axes[1].axis('off')
    axes[2].imshow(prep_i, cmap='gray'); axes[2].set_title('Mejorado: CLAHE + Bilateral'); axes[2].axis('off')
    fig.suptitle('Comparacion de preprocesado', fontsize=12)
    fig.tight_layout()
    plt.show()

---
## 6. Extraccion de medidas

Medidas extraidas por pipeline:
- Altura y ancho maximo
- Anchos relativos al 25%, 50%, 75% de la altura
- Ancho de cuello y base
- Ratio Ancho/Alto y Cuello/Base

In [ ]:
if not images:
    print('Sin imagenes.')
else:
    contour_b = get_bottle_contour(mask_b)
    contour_i = get_bottle_contour(mask_i)
    measures_b = measure_bottle(mask_b, contour_b)
    measures_i = measure_bottle(mask_i, contour_i)

    df_measures = format_measures_table(measures_b, measures_i)
    print(f'Medidas para: {demo["name"]}')
    display(df_measures.style.set_caption('Medidas geometricas de la botella'))

In [ ]:
# Visualizacion con medidas anotadas
if images:
    vis_b = annotate_measurements(gray, measures_b, contour_b)
    vis_i = annotate_measurements(gray, measures_i, contour_i)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(cv2.cvtColor(vis_b, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Medidas BASELINE')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(vis_i, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Medidas MEJORADO')
    axes[1].axis('off')
    fig.suptitle(f'Anotacion de medidas — {demo["name"]}', fontsize=12)
    fig.tight_layout()
    plt.show()

---
## 7. Evaluacion comparativa (todas las imagenes)

In [ ]:
if not images:
    print('Sin imagenes para evaluar.')
else:
    all_results = []
    for img_dict in images:
        name = img_dict['name']
        gray_img = img_dict['image']
        print(f'  Procesando {name} ...')
        b_res = run_baseline(gray_img)
        i_res = run_improved(gray_img)
        all_results.append({'name': name, 'gray': gray_img,
                             'baseline': b_res, 'improved': i_res})
        print(f'    Baseline: {b_res["time_s"]}s  |  Mejorado: {i_res["time_s"]}s')

    df_summary = build_summary_table(all_results)
    print('\nTabla resumen generada.')

In [ ]:
# Mostrar tabla resumen completa
if images:
    cols_show = [
        'imagen',
        'base_height_px', 'impr_height_px',
        'base_width_max_px', 'impr_width_max_px',
        'base_ratio_w_h', 'impr_ratio_w_h',
        'base_ratio_neck_base', 'impr_ratio_neck_base',
        'base_edge_continuity', 'impr_edge_continuity',
        'base_time_s', 'impr_time_s',
        'iou_masks',
    ]
    display(df_summary[cols_show].style
            .format(precision=3)
            .set_caption('Tabla resumen — Baseline vs Mejorado'))

In [ ]:
# Estadisticas descriptivas
if images:
    print_stats(df_summary)

In [ ]:
# Grafica 1: Comparacion de medidas geometricas (barras)
if images:
    out_bar = os.path.join(OUTPUT_FOLDER, 'comparacion_medidas.png')
    fig = plot_comparison_bar(df_summary, output_path=out_bar)
    plt.show()

In [ ]:
# Grafica 2: Calidad de borde y tiempo de procesamiento
if images:
    out_quality = os.path.join(OUTPUT_FOLDER, 'calidad_borde.png')
    fig = plot_edge_quality(df_summary, output_path=out_quality)
    plt.show()

In [ ]:
# Grafica 3: Overlay de contornos para cada imagen
if images:
    for res in all_results:
        out_overlay = os.path.join(OUTPUT_FOLDER,
                                    f'overlay_{os.path.splitext(res["name"])[0]}.png')
        fig = plot_contour_overlay(
            res['gray'], res['baseline'], res['improved'],
            title=res['name'], output_path=out_overlay
        )
        plt.show()

In [ ]:
# Guardar tabla resumen como CSV y HTML
if images:
    save_outputs(df_summary, OUTPUT_FOLDER)
    print(f'\nOutputs guardados en: {OUTPUT_FOLDER}')

---
## 8. Conclusiones y siguientes pasos

### Resultados obtenidos

| Aspecto | Baseline | Mejorado |
|---------|----------|----------|
| Preprocesado | Gaussiano estandar | CLAHE + Bilateral + supresion de reflejos |
| Deteccion de bordes | Canny + cierre 5x5 | Canny L2 + cierre 7x7 (2 iter) + dilatacion |
| Mascara | Relleno del contorno mas grande | Morfologia robusta + validacion keypoints |
| Tiempo | Mas rapido | Moderadamente mas lento |

### Interpretacion de metricas
- **`edge_continuity`**: valores altos indican bordes mas completos y conectados (mejor para contornos de botella).
- **`iou_masks`**: cercano a 1.0 indica que ambos pipelines detectan practicamente la misma region.
- **`ratio_neck_base`**: caracteristica clave para distinguir tipos de botella (estrecha/ancha).

### Siguientes pasos recomendados
1. **Calibracion metrica**: relacionar pixeles con milimetros usando una referencia fisica conocida (escala en la escena).
2. **Segmentacion semantica**: reemplazar Canny por una red U-Net entrenada en botellas para mayor robustez.
3. **Clasificacion por tipo**: usar las medidas para clasificar automaticamente entre modelos de botella.
4. **Control de calidad**: definir tolerancias por medida y marcar botellas fuera de especificacion.